# Exercise 1: Build Your Own Adstock

Geometric adstock spreads a media impulse over several periods, with each lag weighted by
`alpha` ($\alpha$, the retention rate) raised to the lag.
This exercise asks you to implement the transform yourself, without calling
`mmm_book.adstock.geometric_adstock`, and check your result against it.

In [1]:
import numpy as np

from mmm_book import RANDOM_SEED
from mmm_book.adstock import geometric_adstock
from mmm_book.plotting import mmm_style

mmm_style()
rng = np.random.default_rng(RANDOM_SEED)

## Task

Implement `my_geometric_adstock(x, alpha, l_max=12, normalize=True)`.
It should spread each value of `x` forward over `l_max` periods, with lag `l` weighted by
`alpha ** l` (the book's $w_l = \alpha^l$; the book uses $k$ for the half-saturation point).
When `normalize=True`, rescale the lag weights to sum to one before applying them, so the
adstocked series is not inflated relative to the raw input.
You can implement this with an explicit loop over lags, or with a convolution.
Either approach is fine as long as the output matches `mmm_book.adstock.geometric_adstock`
on the same inputs.

In [2]:
def my_geometric_adstock(x, alpha, l_max=12, normalize=True):
    """Geometric adstock, implemented from scratch with a convolution.

    Lag l is weighted by alpha ** l. Convolving x with those weights and keeping the
    first len(x) entries of the result gives exactly the same lag-weighted sum that a loop
    over lags would build, since np.convolve's full-mode output at index n is the sum of
    weights[l] * x[n - l] over all valid l, which is the definition of adstock at week n.
    """
    x = np.asarray(x, dtype=float)
    weights = alpha ** np.arange(l_max)
    if normalize:
        weights = weights / weights.sum()
    return np.convolve(x, weights)[: len(x)]

## Test series

A bursty spend series, the same shape of input adstock is meant to smooth, used to check
your implementation.

In [3]:
n_weeks = 52
spend = np.zeros(n_weeks)
burst_weeks = [4, 5, 6, 20, 21, 34]
spend[burst_weeks] = rng.uniform(80, 120, size=len(burst_weeks))

alpha = 0.6
l_max = 12

my_result = my_geometric_adstock(spend, alpha=alpha, l_max=l_max, normalize=True)
reference_result = geometric_adstock(spend, alpha=alpha, l_max=l_max, normalize=True)
print(f"my_geometric_adstock output shape: {my_result.shape}")

my_geometric_adstock output shape: (52,)


## Self-check

In [4]:
matches = np.allclose(my_result, reference_result, atol=1e-8)
print(f"matches mmm_book.adstock.geometric_adstock: {matches}")
if matches:
    print("Self-check: PASS")
else:
    print("Self-check: FAIL (stub still in place, or implementation not yet correct)")

matches mmm_book.adstock.geometric_adstock: True
Self-check: PASS
